# Read-level SINTAX analysis across folders
This notebook parses all taxonomic ranks and their recorded confidence values,
recomputes the lowest assignment at confidence >=0.8, joins raw-parent primer hits,
and produces per-folder and combined tab-delimited read/species tables.
Contamination means confident occurrence in >=80% of distinct analysed PCR_plate
codes, assessed separately at each rank. Missing folders are explicitly recorded
and excluded from the denominator. This is a prevalence flag, not proof of contamination.
Species summary medians and read counts use species confidence >=0.8; candidate-only
species remain with count zero and blank confident median. Raw parent counts are
also reported separately from cleaned query counts.


## Primer naming
The sequence-based dictionary maps original MFT names (keys), such as
`forward_tagged_primer_1` and `reverse_tagged_primer_1`, to PCR names (values),
`bc1002` and `bc1100rc`, respectively. These numbers are not interchangeable.
Only the dictionary values are exported in biological tables: `forward_PCR_primer`
and `reverse_PCR_primer` at read/primer-pair level, and `forward_primers_detected`
and `reverse_primers_detected` in aggregated summaries. Lists contain unique,
semicolon-separated values. The mapping is saved separately as JSON for provenance.


In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code/sintax_analysis.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'code'))
ANALYSIS_DIR = Path(os.environ.get('SINTAX_ANALYSIS_ROOT', str(PROJECT_ROOT / 'data/sintax_analysis')))

from sintax_analysis import run_analysis
from datetime import datetime
MERGED_DIR = PROJECT_ROOT / 'data/mergedseq5'
SINTAX_ROOT = PROJECT_ROOT / 'data/sintax'
FOLDERS = os.environ.get('SINTAX_FOLDERS', ','.join(f'barcode{i:02d}' for i in range(1,25))).split(',')
PRIMER_HITS = Path(os.environ.get('PRIMER_BEST_HITS', str(PROJECT_ROOT / 'data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260731.txt')))
PLATE_MAP = PROJECT_ROOT / 'saproxDNA/species_info_seq5/seq5_pcr_to_pools.tsv'
# Output must be a new directory; analysis never overwrites a previous run.
print('Output:', ANALYSIS_DIR)


In [ ]:
metadata = run_analysis(project_root=PROJECT_ROOT, merged_dir=MERGED_DIR,
    sintax_root=SINTAX_ROOT, output_dir=ANALYSIS_DIR, primer_hits=PRIMER_HITS,
    plate_map=PLATE_MAP, folders=FOLDERS, cutoff=0.8, prevalence=0.8,
    allow_missing=True)


In [ ]:
manifest = pd.read_csv(ANALYSIS_DIR / 'input_manifest.tsv', sep='	')
summary = pd.read_csv(ANALYSIS_DIR / 'barcode_analysis_summary.tsv', sep='	')
prevalence = pd.read_csv(ANALYSIS_DIR / 'taxon_plate_prevalence.tsv', sep='	')
species = pd.read_csv(ANALYSIS_DIR / 'all_species_summary.tsv', sep='	')
display(manifest)
display(summary)
display(prevalence[prevalence['contamination'].eq('contamination')])
display(species.sort_values('read_count', ascending=False).head(40))
